# fusion_dev1000 — DUNG CAI CAN, do bang PRECISION

**Muc dich:** sinh diem CE cho **dev1000 o K=50**, dung cau hinh bai da nop.
Khong phai de tim huong moi — de **do duoc** cac huong tiep theo.

Ngay 07/09 thu 8 huong tren dev300, **khong huong nao dat p < 0.05**: moi can gat con lai dang
1–2 diem, sai so ghep cap cua dev300 la ~1,5 diem. dev1000 ha san nhieu ~1,8 lan.

## ⚠️ HAI THAY DOI SO VOI BAN CU — doc ky

| | ban cu | ban nay |
|---|---|---|
| `K_CHUNK` | 20 | **50** — khop bai nop 0.702. `ce_deep` cham o K=20 **khong so duoc** voi K=50 |
| o do (cell 12) | recall@5 + quet `n` | **precision khi nop 1 id** = dung o hang 1 |

`blend` / `n` da chet: nop 1 id thi chi con mot suat, luon lay hang 1.
Da chung minh m=1 luon toi uu — precision theo so id nop: **0.7100** / 0.4300 / 0.3056 / 0.1933.

## Chi phi (nhip that 12,9 doan/s, do trong luu tru)

| luot | MODE | M | doan | GPU |
|---|---|---|---|---|
| **A** | `dev1000_a` | 10 | tang1 99k + tang2 209k | **~6,6h** |
| **B** | `dev1000_b` | 20 (skip=10) | tang2 209k | **~4,5h** |

Tong ~11–12h chia hai phien (tran Kaggle 12h/phien, 30h/tuan).
Chay A xong **tai `outputs/` ve va upload lai lam dataset**, roi doi `MODE = "dev1000_b"`.

## Nguong dat TRUOC khi chay (moc: dev300@K20 = 0.7100 · public@K50 = 0.702)

| `max` tren dev1000 ra | ket luan |
|---|---|
| < 0.60 | nghi BUG — tut qua sau so voi public |
| 0.60 – 0.68 | dev1000 kho hon dev300, dung nhu du doan |
| **0.68 – 0.75** | **khop public → dev1000 la can dung, dung cho moi thi nghiem sau** |
| > 0.75 | nghi lot cau dev — kiem tra truoc khi tin |

In [ ]:
!pip install -q sentence-transformers

In [ ]:
# ===== Bước 0: cấu hình + đường dẫn + dấu vân tay =====
import os, sys, json, time, hashlib
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

MODE = "dev1000_a"    # "dev"|"dev1000_a"|"dev1000_b"|"public_a"|"public_b"  <-- ĐỔI ĐÚNG DÒNG NÀY
                      # dev1000: kiểm n=3 và M=20 có còn là đỉnh trên mẫu gấp 3 hay không.
                      # Chạy _a trước (M=10), tải outputs/ lên dataset, rồi chạy _b (skip=10 -> M=20).
RO   = "_matchEmbedded"   # "" = rổ fusion cũ · "_matchEmbedded" = rổ D giao 23/08
TAG  = "matchEmb"          # tên NGẮN dán vào file nộp. Chạy public_b thì đổi thành "matchEmb_M20"
                          # rổ mới: recall@50 0.9817->0.9883 · recall@5 thô +3,83
M_DOC, K_CHUNK, TOPK = 20, 50, 5   # K=50: KHOP cau hinh bai nop 0.702. K=20 KHONG so duoc.
MOC_CU = 0.9350       # max n=1 trên rổ fusion CŨ — mốc phải vượt
NS = (0, 1, 2, 3, 4)  # rổ đổi thì ĐỈNH n DỜI (kết luận 3, 23/08). Quét trọn, 0 giờ GPU

INPUT_DIR = next(p for p in ("/kaggle/input/project-ir",
                             "/kaggle/input/datasets/locdovan211/project-ir")
                 if os.path.isdir(p))
CTX_DIR = next(p for p in (f"{INPUT_DIR}/selected-contexts/selected-contexts",
                           f"{INPUT_DIR}/selected-contexts")
               if os.path.isdir(p) and any(f.startswith("context_") for f in os.listdir(p)))
OUT = "/kaggle/working/outputs"; os.makedirs(OUT, exist_ok=True)
sys.path.append(INPUT_DIR)

CFG = {                       # (file ứng viên, file câu hỏi, skip, tên file ra)
 "dev":      (f"{INPUT_DIR}/fusion_rrf_top50_dev_1000{RO}.json", f"{INPUT_DIR}/dev_300_locked.json",     0,  f"scores_dev300_fusion_M20_K20{RO}.json"),
 "dev1000_a":(f"{INPUT_DIR}/fusion_rrf_top50_dev_1000{RO}.json", f"{INPUT_DIR}/dev_1000_locked.json",   0,  f"scores_dev1000_fusion_M10_K50{RO}.json"),
 "dev1000_b":(f"{INPUT_DIR}/fusion_rrf_top50_dev_1000{RO}.json", f"{INPUT_DIR}/dev_1000_locked.json",  10,  f"scores_dev1000_fusion_M20_K50{RO}.json"),
 "public_a": (f"{INPUT_DIR}/fusion_rrf_top50_public{RO}.json",   f"{INPUT_DIR}/public-official.json",     0,  f"scores_public_fusion_M10_K20{RO}.json"),
 "public_b": (f"{INPUT_DIR}/fusion_rrf_top50_public{RO}.json",   f"{INPUT_DIR}/public-official.json",    10,  f"scores_public_fusion_M20_K20{RO}.json"),
 "public_t1":(f"{INPUT_DIR}/fusion_rrf_top50_public{RO}.json",   f"{INPUT_DIR}/public-official.json",     0,  f"scores_public_fusion_TANG1{RO}.json"),   # CHỈ tầng 1, có checkpoint
}
CAND_F, Q_F, SKIP, OUT_NAME = CFG[MODE]
M_RUN = 10 if MODE in ("public_a", "dev1000_a") else M_DOC   # tách 2 lượt cho dưới trần 12h

for f in ("deep_chunk.py", "rerank_from_d.py", "rerank.py"):
    b = open(f"{INPUT_DIR}/{f}", "rb").read()
    print(f"{f:22} {len(b):>6} bytes  {hashlib.sha256(b).hexdigest()[:12]}")

import deep_chunk as DC
from rerank import load_reranker
from rerank_from_d import blend_bm25_first, score_docs_from_d
DC.MERGE_CHARS = 1800                      # cùng cấu hình bài chốt 0.8871

assert hasattr(DC, "MERGE_CHARS"), "deep_chunk.py là BẢN CŨ — upload lại"
assert os.path.isfile(CAND_F), f"CHƯA UPLOAD {CAND_F}"
print(f"\nMODE={MODE} · M={M_RUN} · SKIP={SKIP} · CTX={CTX_DIR}")

## Bước 1 — Nạp rổ fusion, kiểm khớp câu hỏi

`rrf_score` vào khoá `bm25`: `blend_bm25_first` sẽ ép 2 suất đầu theo thứ tự **fusion**
chứ không phải BM25 thô. Đó là chỗ ăn điểm — fusion xếp recall@5 = 0.8733 so với 0.7533.

In [ ]:
cand = json.load(open(CAND_F, encoding="utf-8"))
qsrc = json.load(open(Q_F, encoding="utf-8"))
questions = {q: v["question"] for q, v in qsrc.items() if q in cand}
assert len(questions) == len(qsrc), f"THIẾU {len(qsrc)-len(questions)} câu trong rổ ứng viên"

rrf = {q: {str(c["doc_id"]): float(c["rrf_score"]) for c in cand[q]} for q in questions}
nc = [len(cand[q]) for q in questions]
print(f"{len(questions)} câu · {min(nc)}-{max(nc)} ứng viên/câu")
print(f"tổng chunk tầng 1: {sum(len(c['top_chunks']) for q in questions for c in cand[q]):,}")

if MODE.startswith("dev"):
    gold = {q: {str(x) for x in qsrc[q]["answer"]} for q in questions}
    fr = {q: [str(c["doc_id"]) for c in cand[q]] for q in questions}
    for k in (5, 50):
        r = sum(len(gold[q] & set(fr[q][:k]))/len(gold[q]) for q in gold)/len(gold)
        print(f"  TRẦN rổ: recall@{k} = {r:.4f}" + ("   <- không ai vượt được" if k == 50 else ""))

## Bước 2 — Tầng 1: chấm `top_chunks` của D (rổ 50 nên rẻ bằng nửa lần trước)

In [ ]:
score_fn = load_reranker("AITeamVN/Vietnamese_Reranker", device="cuda")

# public_b nạp lại tầng 1 của public_a -> khỏi chấm lại ~100k đoạn (~2h)
BASE = "dev1000" if MODE.startswith("dev1000") else "public"
PREV = f"{INPUT_DIR}/scores_{BASE}_fusion_M10_K50{RO}.json"   # lượt B nạp output lượt A
if MODE.endswith("_b") and os.path.isfile(PREV):
    scores = json.load(open(PREV, encoding="utf-8"))
    nd = [sum(1 for v in s.values() if "ce_deep" in v) for s in scores.values()]
    assert min(nd) == max(nd) == 10, f"file lượt A phải có ĐÚNG 10 ce_deep/câu, thấy {min(nd)}-{max(nd)}"
    print(f"nạp lại tầng 1 + ce_deep hạng 1-10 từ lượt A: {len(scores)} câu — BỎ QUA Bước 2")
else:
    assert not MODE.endswith("_b"), f"CHƯA UPLOAD {PREV} — lượt B cần output của lượt A"
    p1 = f"{OUT}/tang1_{OUT_NAME}"
    # nối tiếp lượt trước nếu đã upload file dở (hết quota giữa chừng thì chạy lại là tiếp)
    RESUME = f"{INPUT_DIR}/tang1_{OUT_NAME}"
    scores = json.load(open(RESUME, encoding="utf-8")) if os.path.isfile(RESUME) else {}
    todo = [q for q in questions if q not in scores]
    print(f"tầng 1: đã có {len(scores)} câu, cần chạy {len(todo)} câu")

    t0 = time.time()
    for i, q in enumerate(todo, 1):
        ce = score_docs_from_d(questions[q], cand[q], score_fn)
        scores[q] = {d: {"ce": float(s), "bm25": rrf[q].get(d, 0.0)} for d, s in ce.items()}
        if i % 100 == 0 or i == len(todo):
            json.dump(scores, open(p1, "w", encoding="utf-8"), ensure_ascii=False)  # checkpoint
            el = time.time() - t0
            print(f"  {i}/{len(todo)} | {el/60:.1f} phút | còn ~{el/i*(len(todo)-i)/60:.1f} phút | đã lưu", flush=True)

    json.dump(scores, open(p1, "w", encoding="utf-8"), ensure_ascii=False)
    print(f"ĐÃ LƯU {p1} ({len(scores)}/{len(questions)} câu) — TẢI VỀ dù phần dưới hỏng")
    if MODE == "public_t1":
        print("\nMODE=public_t1: DỪNG Ở ĐÂY. Upload file trên lên dataset rồi chạy public_a.")
        raise SystemExit(0)

## Bước 3 — ĐẾM trước khi chấm tầng 2 (quy tắc 6)

Vượt 6h thì dừng: hạ `K_CHUNK` xuống 12, hoặc với đề thi thì tách `public_a` / `public_b`.

In [ ]:
t0 = time.time()
n2 = DC.count_deep_chunks(questions, scores, CTX_DIR, M_RUN, K_CHUNK, skip=SKIP)
print(f"tầng 2: {n2:,} đoạn ({n2/len(questions):.0f}/câu) | băm+đếm {time.time()-t0:.0f}s")
print(f"ước {n2/12.9/3600:.1f}h @12,9 đoạn/s  ·  {n2/6.5/3600:.1f}h nếu chậm 2x")
assert n2 < 450_000, "quá nhiều — kiểm M_RUN / K_CHUNK / MERGE_CHARS"

## Bước 4 — Tầng 2: đọc sâu top-M theo hạng CE

In [ ]:
t0 = time.time()
scores = DC.deepen_all(questions, scores, CTX_DIR, score_fn, M_RUN, K_CHUNK, skip=SKIP)
el = time.time() - t0
print(f"tầng 2 xong {el/60:.0f} phút | nhịp thật {n2/el:.1f} đoạn/s")

p2 = f"{OUT}/{OUT_NAME}"
json.dump(scores, open(p2, "w", encoding="utf-8"), ensure_ascii=False)
print(f"ĐÃ LƯU {p2} — TẢI VỀ TRƯỚC KHI ĐÓNG PHIÊN")

## Bước 5 — Đo (dev) / đóng gói bài nộp (đề thi)

In [ ]:
order = {q: [d for d, _ in sorted(rrf[q].items(), key=lambda x: -x[1])] for q in questions}

if MODE.startswith("dev"):
    # ============================================================================
    # DO DO CHINH THUC = PRECISION, nop DUNG 1 id/cau.
    # Xac nhan 07/09 bang bai that: 1 id -> precision 0.702 (5 id -> 0.1958).
    # precision(1 id) = ti le cau ma van ban HANG 1 nam trong gold.
    # `blend`/`n` KHONG con y nghia: chi con mot suat, luon lay hang 1.
    # ============================================================================
    P1 = lambda v: sum(DC.rank_by(scores[q], v)[0] in gold[q] for q in questions) / len(questions)
    R5 = lambda v: sum(len(gold[q] & set(DC.rank_by(scores[q], v)[:5])) / len(gold[q])
                       for q in questions) / len(questions)

    print(f"{'bien the':10s}{'PRECISION (1 id)':>18}{'recall@5 (tham khao)':>24}")
    print("-" * 52)
    tab = {}
    for v in DC.VARIANTS:
        tab[v] = P1(v)
        print(f"{v:10s}{tab[v]:>18.4f}{R5(v):>24.4f}")

    best_v = max(tab, key=tab.get)
    print(f"\nDINH tren {len(gold)} cau: bien the '{best_v}' -> {tab[best_v]:.4f}")
    print(f"  'max' (cau hinh DANG NOP)  -> {tab['max']:.4f}")

    # tran cua ro: khong bo xep hang nao vuot duoc
    tran = sum(bool(gold[q] & set(order[q])) for q in questions) / len(questions)
    print(f"  tran cua ro 50 van ban     -> {tran:.4f}   (du dia con lai {tran-tab['max']:+.4f})")

    json.dump({v: tab[v] for v in tab}, open(f"{OUT}/precision1_{MODE}.json", "w"))

    if MODE.startswith("dev1000"):
        # NGUONG DAT TRUOC KHI CHAY — moc dev300@K20 = 0.7100, public@K50 = 0.702
        got = tab["max"]
        print("\n" + "=" * 62)
        if   got < 0.60: print("!!! TUT QUA SAU so voi 0.702 tren public = NGHI BUG. Soi lai cach nap ro.")
        elif got < 0.68: print("dev1000 KHO HON dev300 (da biet: ro recall@5 lech 3,16 diem). Binh thuong.")
        elif got <= 0.75: print("KHOP public 0.702 -> dev1000 la CAN DUNG. Dung no cho moi thi nghiem sau.")
        else: print("CAO HON public nhieu -> nghi lot cau dev vao dau do. Kiem tra truoc khi tin.")
        print("=" * 62)
        print("\nBAY GIO CO CAN. Moi hieu ung 1-2 diem deu do duoc. Viec tiep: BO PHAN XU CAP.")
        raise SystemExit(0)
else:
    import zipfile
    dev1k = json.load(open(f"{INPUT_DIR}/dev_1000_locked.json", encoding="utf-8"))
    # NOP DUNG 1 ID — da chung minh: m=1 luon toi uu voi precision (0.7100 / 0.4300 / 0.3056 / 0.1933)
    sub = {q: {"answer": [DC.rank_by(scores[q], "max")[0]]} for q in questions}
    assert len(sub) == 1000 and set(sub) == set(qsrc), "qid khong khop de thi"
    assert not (set(sub) & set(dev1k)), "LOT CAU DEV — dung"
    assert all(len(v["answer"]) == 1 for v in sub.values()), "phai dung 1 id/cau"
    z = f"{OUT}/submission_{TAG}_TOP1.zip"
    json.dump(sub, open(f"{OUT}/submission.json", "w", encoding="utf-8"), ensure_ascii=False)
    with zipfile.ZipFile(z, "w", zipfile.ZIP_DEFLATED) as f:
        f.write(f"{OUT}/submission.json", "submission.json")
    print(f"OK {z}  ({len(sub)} cau, moi cau 1 id)")
    print("MOC PHAI VUOT: precision 0.702")